# Fitting a Logistic Regression Model - Lab

## Introduction

In the last lesson you were given a broad overview of logistic regression. This included an introduction to two separate packages for creating logistic regression models. In this lab, you'll be investigating fitting logistic regressions with `statsmodels`. For your first foray into logistic regression, you are going to attempt to build a model that classifies whether an individual survived the [Titanic](https://www.kaggle.com/c/titanic/data) shipwreck or not (yes, it's a bit morbid).


## Objectives

In this lab you will: 

* Implement logistic regression with `statsmodels` 
* Interpret the statistical results associated with model parameters

## Import the data

Import the data stored in the file `'titanic.csv'` and print the first five rows of the DataFrame to check its contents. 

In [3]:
# Import the data
import pandas as pd

df = pd.read_csv('titanic.csv')

# Displaying the first five rows
print(df.head())

   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123        S  
4      0            373450   8.0500   NaN        S  


## Define independent and target variables

Your target variable is in the column `'Survived'`. A `0` indicates that the passenger didn't survive the shipwreck. Print the total number of people who didn't survive the shipwreck. How many people survived?

In [5]:
# Total number of people who survived/didn't survive
# Total number of people who didn't survive (Survived == 0) and survived (Survived == 1)
print(df['Survived'].value_counts())

Survived
0    549
1    342
Name: count, dtype: int64


Only consider the columns specified in `relevant_columns` when building your model. The next step is to create dummy variables from categorical variables. Remember to drop the first level for each categorical column and make sure all the values are of type `float`: 

In [7]:
# Create dummy variables
relevant_columns = ['Pclass', 'Age', 'SibSp', 'Fare', 'Sex', 'Embarked', 'Survived']
dummy_dataframe = df[relevant_columns]

dummy_dataframe.shape

(891, 7)

Did you notice above that the DataFrame contains missing values? To keep things simple, simply delete all rows with missing values. 

> NOTE: You can use the [`.dropna()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.dropna.html) method to do this. 

In [9]:
# Drop missing rows
dummy_dataframe = dummy_dataframe.dropna()
dummy_dataframe.shape

(712, 7)

Finally, assign the independent variables to `X` and the target variable to `y`: 

In [11]:
# Split the data into X and y
y = dummy_dataframe['Survived']
X = dummy_dataframe.drop('Survived', axis=1)

## Fit the model

Now with everything in place, you can build a logistic regression model using `statsmodels` (make sure you create an intercept term as we showed in the previous lesson).  

> Warning: Did you receive an error of the form "LinAlgError: Singular matrix"? This means that `statsmodels` was unable to fit the model due to certain linear algebra computational problems. Specifically, the matrix was not invertible due to not being full rank. In other words, there was a lot of redundant, superfluous data. Try removing some features from the model and running it again.

In [13]:
# Build a logistic regression model using statsmodels

import statsmodels.api as sm

relevant_columns = ['Pclass', 'Age', 'SibSp', 'Fare', 'Sex', 'Embarked', 'Survived']
df_subset = df[relevant_columns]

# Dropping missing values
df_subset = df_subset.dropna()

# Creating dummy variables
dummy_dataframe = pd.get_dummies(df_subset, drop_first=True)

# Confirming everything is numeric
print(dummy_dataframe.dtypes)

Pclass          int64
Age           float64
SibSp           int64
Fare          float64
Survived        int64
Sex_male         bool
Embarked_Q       bool
Embarked_S       bool
dtype: object


In [14]:
# Splitting into X and y
y = dummy_dataframe['Survived'].astype(float)
X = dummy_dataframe.drop(columns='Survived').astype(float)

In [15]:
# Build a logistic regression model using statsmodels

import statsmodels.api as sm

X = sm.add_constant(X)  # Adding intercept
model = sm.Logit(y, X)
result = model.fit()

Optimization terminated successfully.
         Current function value: 0.444229
         Iterations 6


## Analyze results

Generate the summary table for your model. Then, comment on the p-values associated with the various features you chose.

In [17]:
# Summary table
print(result.summary())

                           Logit Regression Results                           
Dep. Variable:               Survived   No. Observations:                  712
Model:                          Logit   Df Residuals:                      704
Method:                           MLE   Df Model:                            7
Date:                Sat, 03 May 2025   Pseudo R-squ.:                  0.3417
Time:                        16:30:01   Log-Likelihood:                -316.29
converged:                       True   LL-Null:                       -480.45
Covariance Type:            nonrobust   LLR p-value:                 5.360e-67
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.6378      0.633      8.901      0.000       4.396       6.879
Pclass        -1.2102      0.163     -7.427      0.000      -1.530      -0.891
Age           -0.0433      0.008     -5.263      0.0

### Your comments here

**Summary Table (P-Values):**
- A p-value below 0.05 typically indicates statistical significance.

1. Significant predictors (p < 0.05):
- Pclass (p = 0.000): Negative coefficient → Lower class reduces survival odds.

- Age (p = 0.000): Negative → Older passengers were less likely to survive.

- SibSp (p = 0.002): Negative → Having more siblings/spouses aboard reduced survival odds.

- Sex_male (p = 0.000): Strong negative effect → Males were far less likely to survive.

- Intercept (const): Statistically significant, but mainly used for model baseline.

2. Not statistically significant (p > 0.05):
- Fare (p = 0.635): Contribution to survival is not significant after controlling for other factors.

- Embarked_Q (p = 0.173) and Embarked_S (p = 0.135): Not significant.

## Level up (Optional)

Create a new model, this time only using those features you determined were influential based on your analysis of the results above. How does this model perform?

In [20]:
# Your code here
# Dropping the non-significant predictors and re-fitting the model

# Selecting only influential features
X_reduced = dummy_dataframe[['Pclass', 'Age', 'SibSp', 'Sex_male']].astype(float)
y = dummy_dataframe['Survived'].astype(int)

# Adding a constant 
X_reduced = sm.add_constant(X_reduced)

# Fitting the model
model_reduced = sm.Logit(y, X_reduced)
result_reduced = model_reduced.fit()

print(result_reduced.summary())

Optimization terminated successfully.
         Current function value: 0.446755
         Iterations 6
                           Logit Regression Results                           
Dep. Variable:               Survived   No. Observations:                  712
Model:                          Logit   Df Residuals:                      707
Method:                           MLE   Df Model:                            4
Date:                Sat, 03 May 2025   Pseudo R-squ.:                  0.3379
Time:                        16:30:01   Log-Likelihood:                -318.09
converged:                       True   LL-Null:                       -480.45
Covariance Type:            nonrobust   LLR p-value:                 5.015e-69
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.5908      0.543     10.288      0.000       4.526       6.656
Pclass        -1.3139      0.

### Your comments here

The new model with the reduced set of features is performing well:
- Pseudo R-squared: 0.3379 – The model explains about 34% of the variability in the target variable (Survived).
- Log-Likelihood: -318.09 – This is lower than the previous model given the reduced number of features.
- LLR p-value: 5.015e-69 – The model is statistically significant (p-value is extremely low).

**Model Comparison:**
- The reduced model still provides a good fit with a comparable pseudo R-squared value (0.3379), indicating it performs almost as well as the full model.
- By removing the less impactful features (such as Fare and Embarked), the model becomes more interpretable without significant loss of accuracy.

## Summary 

Well done! In this lab, you practiced using `statsmodels` to build a logistic regression model. You then interpreted the results, building upon your previous stats knowledge, similar to linear regression. Continue on to take a look at building logistic regression models in Scikit-learn!